In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import spacy
import re
import time
import numpy as np
import time

import ast

In [2]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [3]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

In [4]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [97]:
#function to split words and sentences into different lists
def split_words_sentences(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_alternatives_word.append(row["Alt_split"])
        
        
        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_alternatives_sentence.append(row["Alt_split"])
        
    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Alt_split"])
    
    return df_word, df_sentence
        
    


In [98]:
# read csv file
df_rules = pd.read_csv("rules_08MAR_English_h.csv")

In [99]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1450 entries, 0 to 1449
Data columns (total 15 columns):
Lemma                     1450 non-null object
Word_Type                 1450 non-null object
Priority                  798 non-null object
Category                  1450 non-null object
Primary_subcategory       1450 non-null object
Secondary_subcategory     347 non-null object
Alt_Field                 480 non-null object
Alt_Sg_Replacement        711 non-null object
Alt_Sg_people_first       158 non-null object
Alt_Sg_reframed           298 non-null object
Alt_Pl_collective_noun    47 non-null object
Alt_Pl                    139 non-null object
Alt_Pl_people_first       28 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 170.0+ KB


### Drop duplicates

In [100]:
df_rules = df_rules.drop_duplicates()

In [101]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1449 entries, 0 to 1449
Data columns (total 15 columns):
Lemma                     1449 non-null object
Word_Type                 1449 non-null object
Priority                  798 non-null object
Category                  1449 non-null object
Primary_subcategory       1449 non-null object
Secondary_subcategory     347 non-null object
Alt_Field                 479 non-null object
Alt_Sg_Replacement        711 non-null object
Alt_Sg_people_first       158 non-null object
Alt_Sg_reframed           298 non-null object
Alt_Pl_collective_noun    47 non-null object
Alt_Pl                    139 non-null object
Alt_Pl_people_first       28 non-null object
Alt_Pl_reframed           4 non-null object
3rd_party_alternatives    2 non-null object
dtypes: object(15)
memory usage: 181.1+ KB


### Merge alternatives into one column

In [102]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [103]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives']= df_rules.iloc[:,source_col_loc:source_col_loc+7].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [104]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [105]:
#create a subset of dataframe
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

### Check duplicates&homonyms

In [106]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Alt_split
268,challenge,v,unconscious_bias,agentic,"[question, cross-examine, re-evaluate, object ..."
269,challenge,s,unconscious_bias,agentic,"[test, burning question, complex situation, ta..."
407,direct,v,unconscious_bias,agentic,"[guide, show, point, organize, encourage, assi..."
408,direct,a,unconscious_bias,agentic,"[straightfoward, open, candid]"
448,drive,v,unconscious_bias,agentic,"[invest in, support, work toward, take respons..."
449,drive,s,unconscious_bias,agentic,"[spirit, energy, motivation, appetite, growth ..."
461,elderly,a,unconscious_bias,age_old,[older]
462,elderly,s,unconscious_bias,age_old,"[older adult, older people]"
521,female,s,unconscious_bias,gender_identity,"[person, adult, woman, you, they, person who s..."
522,female,a,unconscious_bias,gender_identity,"[--- only if gender identity is relevant, --- ..."


### Homonyms

In [107]:
# check how many homonyms
df_rules["Lemma"].duplicated(keep = False).sum()

46

In [108]:
df_homonyms = df_rules[df_rules["Lemma"].duplicated(keep = False)]

In [109]:
#check first 5 rows
df_homonyms.head()

,Lemma,Word_Type,Category,Primary_subcategory,Alt_split
268,challenge,v,unconscious_bias,agentic,"[question, cross-examine, re-evaluate, object ..."
269,challenge,s,unconscious_bias,agentic,"[test, burning question, complex situation, ta..."
407,direct,v,unconscious_bias,agentic,"[guide, show, point, organize, encourage, assi..."
408,direct,a,unconscious_bias,agentic,"[straightfoward, open, candid]"
448,drive,v,unconscious_bias,agentic,"[invest in, support, work toward, take respons..."


In [110]:
#save homonyms to csv file after preprocessing
df_homonyms.to_csv("homonyms_words_EN.csv", index=False)

### Remove homonyms from dataframe for the next steps pre-processing

In [111]:
homonyms_list = list(df_rules[df_rules["Lemma"].duplicated()]["Lemma"])

In [112]:
df_rules = df_rules.query("Lemma != @homonyms_list")

### Acronyms (abbreviations)

In [113]:
#check word_type
df_rules["Word_Type"].unique()

array(['abbr', 's', 'i', 'a+s', 'a', 'v', 'v+a+s', 'v+s', 'adv+a', 'v+a',
       'a+a+s', 's,v', 'adv', 'v, a', 's+s'], dtype=object)

In [114]:
df_abbr = df_rules[df_rules["Word_Type"]=='abbr']

In [115]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations_EN.csv", index=False)

### Openly_discriminating

In [116]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [117]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 339 entries, 4 to 1444
Data columns (total 5 columns):
Lemma                  339 non-null object
Word_Type              339 non-null object
Category               339 non-null object
Primary_subcategory    339 non-null object
Alt_split              339 non-null object
dtypes: object(5)
memory usage: 15.9+ KB


In [118]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Alt_split
4,African queen,s,openly_discriminating,homophobia,[-]
6,Aunt Jemima,s,openly_discriminating,racism,[-]
7,Barbie,s,openly_discriminating,racism,[-]
8,Beastie Boy,s,openly_discriminating,antisemitism,[-]
9,Beulah,s,openly_discriminating,racism,[-]


In [119]:
df_open_dis_word, df_open_dis_sentence = split_words_sentences(df_openly_discrim)

In [120]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Alt_split
0,Barbie,s,openly_discriminating,racism,[-]
1,Beulah,s,openly_discriminating,racism,[-]
2,Bin-Laden,s,openly_discriminating,antimuslim,[-]
3,Binladen,s,openly_discriminating,antimuslim,[-]
4,Brownie,s,openly_discriminating,homophobia,[-]


In [121]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words_EN.csv", index=False)

In [122]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences_EN.csv", index=False)

### Style

In [123]:
df_style = df_rules[df_rules["Category"]=="style"]

In [124]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 144 entries, 0 to 1441
Data columns (total 5 columns):
Lemma                  144 non-null object
Word_Type              144 non-null object
Category               144 non-null object
Primary_subcategory    144 non-null object
Alt_split              144 non-null object
dtypes: object(5)
memory usage: 6.8+ KB


In [125]:
df_style_word, df_style_sentence = split_words_sentences(df_style)

In [126]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words_EN.csv", index=False)

In [127]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences_EN.csv", index=False)

### Inclusive

In [128]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [129]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 186 entries, 68 to 1445
Data columns (total 5 columns):
Lemma                  186 non-null object
Word_Type              186 non-null object
Category               186 non-null object
Primary_subcategory    186 non-null object
Alt_split              186 non-null object
dtypes: object(5)
memory usage: 8.7+ KB


In [137]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Alt_split
68,accessibility,s,inclusive,d_and_i,[]
75,affectionate,a,inclusive,communal,[]
76,affinity group,s,inclusive,d_and_i,[]
82,ages,s,inclusive,d_and_i,[]
85,agree,v,inclusive,communal,[]


In [130]:
df_inclusive_word, df_inclusive_sentence = split_words_sentences(df_inclusive)

In [131]:
df_inclusive_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Data columns (total 5 columns):
Lemma                  162 non-null object
Word_Type              162 non-null object
Category               162 non-null object
Primary_subcategory    162 non-null object
Alt_split              162 non-null object
dtypes: object(5)
memory usage: 6.5+ KB


In [135]:
#drop alternatives column
df_inclusive_word=df_inclusive_word.drop(columns=['Alt_split'], axis=1)

In [136]:
df_inclusive_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 162 entries, 0 to 161
Data columns (total 4 columns):
Lemma                  162 non-null object
Word_Type              162 non-null object
Category               162 non-null object
Primary_subcategory    162 non-null object
dtypes: object(4)
memory usage: 5.2+ KB


In [138]:
df_inclusive_sentence = df_inclusive_sentence.drop(columns=['Alt_split'], axis=1)

In [139]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
Lemma                  24 non-null object
Word_Type              24 non-null object
Category               24 non-null object
Primary_subcategory    24 non-null object
dtypes: object(4)
memory usage: 896.0+ bytes


In [140]:
#save to csv file after preprocessing
df_inclusive_word.to_csv("inclusive_words_EN.csv", index=False)

In [141]:
#save to csv file after preprocessing
df_inclusive_sentence.to_csv("inclusive_sentences_EN.csv", index=False)